In [5]:
import os
import pickle
import pandas as pd
import torch
import torch.nn as nn
from TorchCRF import CRF
from tqdm.auto import tqdm

from src.data_converter import bio_to_indices

MODELS_DIR = "../../models/iteration-2/"
SUBMISSIONS_DIR = "../../submissions/iteration-2/"
ARTEFACTS_PATH = os.path.join(MODELS_DIR, "artefacts_v2.pkl")
BEST_MODEL_PATH = os.path.join(MODELS_DIR, "bilstm_v2_best.pth")
SUBMISSION_DATA_PATH = "../../data/raw/submission.csv"
OUTPUT_SUBMISSION_PATH = os.path.join(SUBMISSIONS_DIR, "submission_bilstm_v2.csv")

os.makedirs(SUBMISSIONS_DIR, exist_ok=True)

In [6]:
class CharEmbedding(nn.Module):
    def __init__(self, char_vocab_size, embedding_dim, hidden_dim, dropout_rate=0.25):
        super(CharEmbedding, self).__init__()
        self.embedding = nn.Embedding(char_vocab_size, embedding_dim, padding_idx=0)
        self.lstm = nn.LSTM(input_size=embedding_dim, hidden_size=hidden_dim, num_layers=1, bidirectional=True,
                            batch_first=True)
        self.dropout = nn.Dropout(dropout_rate)
        print("Информация: Модуль CharEmbedding успешно инициализирован.")

    def forward(self, x):
        batch_size, seq_len, word_len = x.size()
        x = x.view(batch_size * seq_len, word_len)
        embedded = self.embedding(x)
        embedded = self.dropout(embedded)
        lstm_out, _ = self.lstm(embedded)
        output = lstm_out.permute(0, 2, 1)
        output = torch.max(output, 2)[0]
        output = output.view(batch_size, seq_len, -1)
        return self.dropout(output)


class BiLSTMCrfForNer(nn.Module):
    def __init__(self, word_vocab_size, word_embedding_dim, char_vocab_size, char_embedding_dim, char_hidden_dim, lstm_hidden_dim, num_tags, dropout_rate=0.33, padding_idx=0):
        super(BiLSTMCrfForNer, self).__init__()
        self.word_embedding = nn.Embedding(num_embeddings=word_vocab_size, embedding_dim=word_embedding_dim, padding_idx=padding_idx)
        self.word_embedding.weight.requires_grad = True
        self.char_embedding = CharEmbedding(char_vocab_size=char_vocab_size, embedding_dim=char_embedding_dim, hidden_dim=char_hidden_dim, dropout_rate=dropout_rate)
        self.embedding_dropout = nn.Dropout(dropout_rate)
        self.lstm = nn.LSTM(input_size=word_embedding_dim + (2 * char_hidden_dim), hidden_size=lstm_hidden_dim, num_layers=2, bidirectional=True, batch_first=True, dropout=dropout_rate if 2 > 1 else 0)
        self.classifier = nn.Linear(2 * lstm_hidden_dim, num_tags)
        self.crf = CRF(num_tags=num_tags, batch_first=True)
        print("Информация: Основная модель BiLSTMCrfForNer успешно инициализирована.")

    def forward(self, word_ids, char_ids, mask, tags=None):
        word_embeds = self.word_embedding(word_ids)
        char_embeds = self.char_embedding(char_ids)
        combined_embeds = torch.cat([word_embeds, char_embeds], dim=-1)
        combined_embeds = self.embedding_dropout(combined_embeds)
        lstm_out, _ = self.lstm(combined_embeds)

        emissions = self.classifier(lstm_out)
        mask = mask.bool()
        if tags is not None:
            loss = -self.crf(emissions, tags, mask=mask, reduction='mean')
            return loss
        else:
            decoded_tags = self.crf.decode(emissions, mask=mask)
            return decoded_tags


In [7]:
from transformers import AutoTokenizer

class NERPipeline:
    def __init__(self, model_path, artefacts_path, tokenizer_name="xlm-roberta-base"):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print(f"Информация: Пайплайн будет использовать устройство: {self.device}")

        print("Информация: Загрузка артефактов...")
        with open(artefacts_path, "rb") as f:
            artefacts = pickle.load(f)
        self.word2id = artefacts["word2id"]
        self.char2id = artefacts["char2id"]
        self.id2tag = artefacts["id2tag"]
        self.tok_norm_fn = lambda t: "".join("0" if c.isdigit() else c for c in t.lower().strip())
        print("Информация: Артефакты успешно загружены.")

        print(f"Информация: Загрузка токенизатора {tokenizer_name}...")
        self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)

        print("Информация: Инициализация архитектуры модели...")
        self.model = BiLSTMCrfForNer(
            word_vocab_size=len(self.word2id),
            char_vocab_size=len(self.char2id),
            num_tags=len(self.id2tag),
            word_embedding_dim=300,
            char_embedding_dim=50,
            char_hidden_dim=50,
            lstm_hidden_dim=256,
            dropout_rate=0.5,
            padding_idx=self.word2id["<PAD>"]
        )

        print(f"Информация: Загрузка весов модели из {model_path}...")
        self.model.load_state_dict(torch.load(model_path, map_location=self.device))
        self.model.to(self.device)
        self.model.eval()
        print("Информация: Модель готова к работе.")

    def _tokenize_and_filter(self, text: str) -> list[str]:
        encoding = self.tokenizer(text, return_offsets_mapping=True)
        tokens = self.tokenizer.convert_ids_to_tokens(encoding["input_ids"])
        offsets = encoding["offset_mapping"]

        filtered_tokens = []
        for token, offset in zip(tokens, offsets):
            if offset != (0, 0):
                filtered_tokens.append(token)
        return filtered_tokens

    def predict(self, text: str) -> list:
        with torch.no_grad():
            tokens = self._tokenize_and_filter(text)

            word_ids = [self.word2id.get(self.tok_norm_fn(token), self.word2id["<UNK>"]) for token in tokens]

            char_ids_per_word = []
            for token in tokens:
                ids = [self.char2id.get(char, self.char2id["<UNK>"]) for char in token]
                char_ids_per_word.append(ids)

            max_word_len = max(len(ids) for ids in char_ids_per_word) if char_ids_per_word else 0
            padded_chars = []
            for ids in char_ids_per_word:
                padded_chars.append(ids + [self.char2id["<PAD>"]] * (max_word_len - len(ids)))

            word_tensor = torch.tensor([word_ids], dtype=torch.long).to(self.device)
            char_tensor = torch.tensor([padded_chars], dtype=torch.long).to(self.device)
            mask_tensor = torch.tensor([[1] * len(tokens)], dtype=torch.bool).to(self.device)

            predictions_ids = self.model(word_tensor, char_tensor, mask_tensor)

            num_tokens = len(tokens)
            predicted_tags = [self.id2tag[tag_id] for tag_id in predictions_ids[0][:num_tokens]]

            return predicted_tags

In [8]:
print("--- Начало процесса предсказания ---")

try:
    pipeline = NERPipeline(model_path=BEST_MODEL_PATH, artefacts_path=ARTEFACTS_PATH)
except FileNotFoundError:
    print(f"Ошибка: Не найдены артефакты модели. Убедитесь, что ноутбук 01_Train_BiLSTM_v1.ipynb успешно отработал.")
    pipeline = None

if pipeline:
    submission_df = pd.read_csv(SUBMISSION_DATA_PATH, sep=";")
    print(f"Загружено {len(submission_df)} записей для предсказания.")

    all_annotations = []
    for index, row in tqdm(submission_df.iterrows(), total=len(submission_df), desc="Предсказание"):
        text = row["sample"]

        predicted_bio_tags = pipeline.predict(text)

        annotations = bio_to_indices(text, predicted_bio_tags)

        all_annotations.append(str(annotations))

    submission_df["annotation"] = all_annotations
    submission_df.to_csv(OUTPUT_SUBMISSION_PATH, sep=";", index=False)

    print("\n--- Процесс предсказания завершен ---")
    print(f"Файл с предсказаниями сохранен в: {OUTPUT_SUBMISSION_PATH}")

--- Начало процесса предсказания ---
Информация: Пайплайн будет использовать устройство: cpu
Информация: Загрузка артефактов...
Информация: Артефакты успешно загружены.
Информация: Загрузка токенизатора xlm-roberta-base...
Информация: Инициализация архитектуры модели...
Информация: Модуль CharEmbedding успешно инициализирован.
Информация: Основная модель BiLSTMCrfForNer успешно инициализирована.
Информация: Загрузка весов модели из ../../models/iteration-2/bilstm_v2_best.pth...
Информация: Модель готова к работе.
Загружено 5000 записей для предсказания.


Предсказание:   0%|          | 0/5000 [00:00<?, ?it/s]


--- Процесс предсказания завершен ---
Файл с предсказаниями сохранен в: ../../submissions/iteration-2/submission_bilstm_v2.csv
